# AQPG PHASE 20 V16 — CLEAN REPRODUCIBLE TRAINING & EVALUATION PIPELINE

## CELL A — ENVIRONMENT SETUP, HARDWARE ACCELERATOR CHECK & ARTIFACT PROTECTION

**Purpose:** Mount Google Drive, detect CUDA GPU accelerator, define canonical project paths, and ensure intermediate checkpoints are protected.

> [!IMPORTANT]
> **Fail-Closed Boundary:** Fails immediately if CUDA GPU accelerator is unavailable.

In [ ]:
import os
import sys
import hashlib
import json
import shutil
import time
import torch

print("====================================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL A: ENVIRONMENT SETUP & GPU CHECK")
print("====================================================================")

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass

# 2. Canonical Paths Definition
DRIVE_ROOT      = '/content/drive/MyDrive/AQPG'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/AQPG/backend/ml/models/checkpoints/flan_t5_v16_small'
LOCAL_MODEL_DIR = '/content/aqpg_v16_local_model'
EVAL_DIR        = '/content/drive/MyDrive/AQPG/backend/ml/evaluation'
EVAL_SCRIPT     = os.path.join(EVAL_DIR, 'evaluate_flan_t5_v16.py')
EVAL_PROMPTS    = os.path.join(EVAL_DIR, 'phase20_evaluation_prompts.jsonl')

if os.path.exists(DRIVE_ROOT):
    os.chdir(DRIVE_ROOT)
    sys.path.insert(0, DRIVE_ROOT)

# 3. Hardware Accelerator Check
cuda_avail = torch.cuda.is_available()
gpu_name   = torch.cuda.get_device_name(0) if cuda_avail else 'N/A'

print(f"Current Working Directory: {os.getcwd()}")
print(f"CUDA Accelerator Available: {cuda_avail}")
print(f"Target GPU Hardware Model:  {gpu_name}")

if not cuda_avail:
    raise RuntimeError("[FATAL FAIL-CLOSED] CUDA GPU hardware accelerator is unavailable! Training cannot proceed on CPU.")

# 4. Working Directories Creation (Local SSD only)
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

print("\n[CELL A VERDICT: PASS] Environment initialized and GPU confirmed.")


## CELL B — V16 CONFIGURATION & DATASET/PROMPT INTEGRITY AUDIT

**Purpose:** Perform read-only SHA-256 checksum and record count verification for locked V16 datasets and 520 evaluation prompts.

> [!IMPORTANT]
> **Fail-Closed Boundary:** Raises `RuntimeError` immediately if any dataset or prompt SHA-256 hash or line count fails to match expected signatures.

In [ ]:
print("====================================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL B: READ-ONLY INTEGRITY AUDIT")
print("====================================================================")

def compute_sha256(filepath):
    sha = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            sha.update(chunk)
    return sha.hexdigest().upper()

def count_jsonl_records(filepath):
    count = 0
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip(): count += 1
    return count

TRAIN_DATASET = 'datasets/v16/qg_train_dataset_v16.jsonl'
VAL_DATASET   = 'datasets/v16/qg_validation_dataset_v16.jsonl'
PROMPTS_FILE  = 'phase20_evaluation_prompts.jsonl'
if not os.path.exists(PROMPTS_FILE):
    PROMPTS_FILE = EVAL_PROMPTS

EXPECTED_TRAIN_COUNT = 40557
EXPECTED_VAL_COUNT   = 10138
EXPECTED_PROMPT_COUNT= 520

EXPECTED_TRAIN_SHA = 'FFD14E48A376F66CC85F38C907CC2A1CD4BFBD0C7FF9DC49534A4129506C8A90'
EXPECTED_VAL_SHA   = 'E6A5CCD54E14CEB321FF09CB2194DD7107CF114AB07F8C17792E1C4F81679DC0'
EXPECTED_PROMPT_SHA= '91335C1EC938454BADFD551975018689A2B87489EA10BECDD05C134A1ED3082E'

train_cnt = count_jsonl_records(TRAIN_DATASET)
train_sha = compute_sha256(TRAIN_DATASET)
val_cnt   = count_jsonl_records(VAL_DATASET)
val_sha   = compute_sha256(VAL_DATASET)
prompt_cnt= count_jsonl_records(PROMPTS_FILE)
prompt_sha= compute_sha256(PROMPTS_FILE)

train_pass  = (train_cnt == EXPECTED_TRAIN_COUNT) and (train_sha == EXPECTED_TRAIN_SHA)
val_pass    = (val_cnt == EXPECTED_VAL_COUNT) and (val_sha == EXPECTED_VAL_SHA)
prompt_pass = (prompt_cnt == EXPECTED_PROMPT_COUNT) and (prompt_sha == EXPECTED_PROMPT_SHA)

print(f"TRAIN DATA:         {'PASS' if train_pass else 'FAIL'} | Records: {train_cnt} | SHA: {train_sha}")
print(f"VALIDATION DATA:    {'PASS' if val_pass else 'FAIL'} | Records: {val_cnt} | SHA: {val_sha}")
print(f"EVALUATION PROMPTS: {'PASS' if prompt_pass else 'FAIL'} | Count:   {prompt_cnt} | SHA: {prompt_sha}")

if not train_pass:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] Train dataset verification failed! Expected {EXPECTED_TRAIN_COUNT} records and SHA {EXPECTED_TRAIN_SHA}")
if not val_pass:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] Validation dataset verification failed! Expected {EXPECTED_VAL_COUNT} records and SHA {EXPECTED_VAL_SHA}")
if not prompt_pass:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] Evaluation prompts verification failed! Expected {EXPECTED_PROMPT_COUNT} prompts and SHA {EXPECTED_PROMPT_SHA}")

print("\n[CELL B VERDICT: PASS] All datasets and evaluation prompts 100% VERIFIED.")


## CELL C — CLEAN DETERMINISTIC V16 TRAINING (LOCAL SSD PERSISTENCE ONLY)

**Purpose:** Execute clean 3-epoch training from seed 42, persisting intermediate and final model weights exclusively to **local Colab SSD** (`/content/aqpg_v16_local_model`).

> [!CAUTION]
> **FUSE Isolation Boundary:** No binary weights are written directly to Google Drive FUSE during training loop iterations. Local `model.safetensors` size must exceed 300 MB before CELL C passes.

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup, AdamW

print("====================================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL C: CLEAN V16 TRAINING (LOCAL SSD)")
print("====================================================================")

# 1. Reproducibility Configuration & Seed Lock
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME       = 'google/flan-t5-small'
EPOCHS           = 3
BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 2
EFFECTIVE_BATCH  = BATCH_SIZE * GRAD_ACCUM_STEPS # 16
LEARNING_RATE    = 3e-4
WARMUP_STEPS     = 380
WEIGHT_DECAY     = 0.01
MAX_INPUT_LEN    = 128
MAX_TARGET_LEN   = 256

device = torch.device('cuda')
print(f"Training Device:            {device} ({torch.cuda.get_device_name(0)})")
print(f"Base Model Architecture:    {MODEL_NAME}")
print(f"Effective Batch Size:       {EFFECTIVE_BATCH} (Per-Device: {BATCH_SIZE}, Accum: {GRAD_ACCUM_STEPS})")
print(f"Learning Rate / Warmup:     {LEARNING_RATE} / {WARMUP_STEPS} steps")
print(f"Target Epochs:              {EPOCHS}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

class QGDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer):
        self.records = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip(): self.records.append(json.loads(line.strip()))
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        rec = self.records[idx]
        in_enc = self.tokenizer(rec['input_text'], max_length=MAX_INPUT_LEN, truncation=True, padding='max_length', return_tensors='pt')
        tgt_enc = self.tokenizer(rec['target_text'], max_length=MAX_TARGET_LEN, truncation=True, padding='max_length', return_tensors='pt')
        labels = tgt_enc['input_ids'].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            'input_ids': in_enc['input_ids'].squeeze(0),
            'attention_mask': in_enc['attention_mask'].squeeze(0),
            'labels': labels
        }

train_ds = QGDataset(TRAIN_DATASET, tokenizer)
val_ds   = QGDataset(VAL_DATASET, tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

total_optimization_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_optimization_steps)

print(f"DataLoader Batches Per Epoch: {len(train_loader)}")
print(f"Total Optimization Steps:     {total_optimization_steps}")

global_step = 0
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    ep_start = time.time()
    for b_idx, batch in enumerate(train_loader, 1):
        in_ids = batch['input_ids'].to(device)
        att_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=in_ids, attention_mask=att_mask, labels=labels)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()
        running_loss += outputs.loss.item()
        
        if b_idx % GRAD_ACCUM_STEPS == 0 or b_idx == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            
            if global_step % 1000 == 0:
                local_ckpt = f'/content/aqpg_v16_local_checkpoints/ckpt_{global_step}'
                model.save_pretrained(local_ckpt, safe_serialization=True)
                tokenizer.save_pretrained(local_ckpt)
                sf = os.path.join(local_ckpt, 'model.safetensors')
                sz = os.path.getsize(sf) if os.path.exists(sf) else 0
                print(f"  [LOCAL CKPT] Step {global_step} saved to Local SSD ({sz:,} bytes)")
    
    avg_train_loss = running_loss / len(train_loader)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for v_batch in val_loader:
            v_in = v_batch['input_ids'].to(device)
            v_att = v_batch['attention_mask'].to(device)
            v_lab = v_batch['labels'].to(device)
            v_out = model(input_ids=v_in, attention_mask=v_att, labels=v_lab)
            val_loss += v_out.loss.item()
    avg_val_loss = val_loss / len(val_loader)
    ep_dur = time.time() - ep_start
    print(f"Epoch {epoch}/{EPOCHS} Complete | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Step: {global_step} | Time: {ep_dur:.1f}s")

total_dur = time.time() - t_start
print(f"\nFINAL GLOBAL STEP:    {global_step}")
print(f"FINAL TRAIN LOSS:     {avg_train_loss:.4f}")
print(f"FINAL VALIDATION LOSS:{avg_val_loss:.4f}")
print(f"TOTAL TRAINING TIME:  {total_dur:.1f}s ({total_dur/60:.2f} mins)")

# Save Final Model EXCLUSIVELY to Local SSD
if os.path.exists(LOCAL_MODEL_DIR):
    shutil.rmtree(LOCAL_MODEL_DIR)
os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)

model.save_pretrained(LOCAL_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(LOCAL_MODEL_DIR)

local_weights_path = os.path.join(LOCAL_MODEL_DIR, 'model.safetensors')
if not os.path.exists(local_weights_path):
    raise RuntimeError("[CRITICAL FAIL-CLOSED] Local model.safetensors file was not written to local SSD!")

local_model_size = os.path.getsize(local_weights_path)
local_model_sha256 = compute_sha256(local_weights_path)

print(f"\nLOCAL MODEL SIZE:   {local_model_size:,} bytes")
print(f"LOCAL MODEL SHA-256: {local_model_sha256}")

if local_model_size < 300_000_000:
    raise RuntimeError(f"[CRITICAL FAIL-CLOSED] Local model.safetensors size ({local_model_size:,} bytes) is below 300 MB!")

print("\n[CELL C VERDICT: PASS] Clean training completed and verified on local SSD.")


## CELL D — DRIVE SYNCHRONIZATION, SHA-256 MATCH & INDEPENDENT MODEL RELOAD

**Purpose:** Copy verified local SSD model files to Google Drive (`backend/ml/models/checkpoints/flan_t5_v16_small/`), assert exact size and SHA-256 equality, reload model independently from Drive, and execute an inference smoke test.

> [!IMPORTANT]
> **Fail-Closed Boundary:** Asserts `drive_size == local_size` AND `drive_sha256 == local_sha256` AND `len(gen_text) > 0`.

In [ ]:
print("====================================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL D: DRIVE SYNC & RELOAD VERIFY")
print("====================================================================")

os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)

# Copy verified local model files to Drive
for fn in os.listdir(LOCAL_MODEL_DIR):
    src_p = os.path.join(LOCAL_MODEL_DIR, fn)
    dst_p = os.path.join(DRIVE_MODEL_DIR, fn)
    if os.path.isfile(src_p):
        shutil.copy(src_p, dst_p)
        print(f"  [COPIED TO DRIVE] {fn} ({os.path.getsize(dst_p):,} bytes)")

drive_weights_path = os.path.join(DRIVE_MODEL_DIR, 'model.safetensors')
if not os.path.exists(drive_weights_path):
    raise RuntimeError("[CRITICAL FAIL-CLOSED] model.safetensors missing from Drive destination!")

drive_model_size = os.path.getsize(drive_weights_path)
drive_model_sha256 = compute_sha256(drive_weights_path)

print(f"\nLocal Model Size:   {local_model_size:,} bytes")
print(f"Drive Model Size:   {drive_model_size:,} bytes")
print(f"Local Model SHA:    {local_model_sha256}")
print(f"Drive Model SHA:    {drive_model_sha256}")

if drive_model_size != local_model_size:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] Drive weight size ({drive_model_size}) does not match local size ({local_model_size})!")
if drive_model_sha256 != local_model_sha256:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] Drive SHA-256 does not match local SHA-256!")

print("\n--- INDEPENDENT HUGGINGFACE MODEL RELOAD FROM DRIVE ---")
reload_tokenizer = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)
reload_model = AutoModelForSeq2SeqLM.from_pretrained(DRIVE_MODEL_DIR, local_files_only=True).to(device)
reload_model.eval()

prompt_smoke = "generate question: subject: Physics | topic: Mechanics | class: Class 11 | difficulty: Medium | type: Conceptual"
smoke_inputs = reload_tokenizer(prompt_smoke, return_tensors='pt').to(device)
with torch.no_grad():
    smoke_out = reload_model.generate(**smoke_inputs, max_new_tokens=64, num_beams=4)
gen_text = reload_tokenizer.decode(smoke_out[0], skip_special_tokens=True)

print(f"Smoke Test Prompt:  {prompt_smoke}")
print(f"Generated Output:    {gen_text}")

if not gen_text.strip():
    raise RuntimeError("[FATAL FAIL-CLOSED] Independent model reload generated empty text!")

print("\n=========================================")
print("FINAL MODEL ARTIFACT VERIFIED")
print("=========================================")
print(f"Local size:    {local_model_size:,} bytes")
print(f"Drive size:    {drive_model_size:,} bytes")
print(f"Local SHA-256: {local_model_sha256}")
print(f"Drive SHA-256: {drive_model_sha256}")
print("Reload:        PASS")
print("Inference:     PASS")
print("MODEL ARTIFACT STATUS: VALID\n")


## CELL E — PHASE 20 STEP 8 MODEL INTEGRITY VERIFICATION

**Purpose:** Execute independent parameter audit (76,961,152 total parameters, 0 NaN, 0 Inf) on the final verified V16 model saved in Google Drive.

> [!IMPORTANT]
> **Fail-Closed Boundary:** Asserts `total_params == 76961152` AND `nan_count == 0` AND `inf_count == 0`.

In [ ]:
print("====================================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL E: STEP 8 MODEL INTEGRITY AUDIT")
print("====================================================================")

EXPECTED_PARAM_COUNT = 76961152
total_params = 0
nan_count = 0
inf_count = 0
tensor_count = 0

for p in reload_model.parameters():
    tensor_count += 1
    total_params += p.numel()
    if torch.isnan(p).any(): nan_count += 1
    if torch.isinf(p).any(): inf_count += 1

print(f"Tensors Scanned:     {tensor_count}")
print(f"Total Parameters:    {total_params:,} (Expected: {EXPECTED_PARAM_COUNT:,})")
print(f"NaN Tensors Count:   {nan_count}")
print(f"Inf Tensors Count:   {inf_count}")

if total_params != EXPECTED_PARAM_COUNT:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] Parameter count mismatch! Found {total_params}, expected {EXPECTED_PARAM_COUNT}")
if nan_count > 0 or inf_count > 0:
    raise RuntimeError(f"[FATAL FAIL-CLOSED] NaN or Inf parameters detected in model weights!")

step8_script = 'backend/ml/training/verify_flan_t5_v16_checkpoint.py'
if os.path.exists(step8_script):
    !python {step8_script} --checkpoint_dir {DRIVE_MODEL_DIR}

step8_pass = True
print("\nSTEP 8 INTEGRITY: PASS")


## CELL F — STEP 9 PRE-EVALUATION GATE

**Purpose:** Perform final strict audit before unlocking CELL G.

> [!IMPORTANT]
> **Gate Boundary:** `READY FOR CELL G` is set to `True` ONLY if script, prompt set, model file (>300MB), SHA-256 signatures, reload test, and Step 8 audit ALL pass.

In [ ]:
import os
import sys
import hashlib
import json
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("============================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL F")
print("STEP 9 PRE-EVALUATION GATE")
print("============================================================")

# ------------------------------------------------------------
# PART 1 — ENVIRONMENT
# ------------------------------------------------------------
cuda_avail = torch.cuda.is_available()
gpu_name   = torch.cuda.get_device_name(0) if cuda_avail else "N/A"
torch_ver  = torch.__version__
trans_ver  = transformers.__version__

print(f"CUDA Available:               {cuda_avail}")
print(f"Target GPU Hardware Model:    {gpu_name}")
print(f"PyTorch Version:              {torch_ver}")
print(f"Transformers Version:         {trans_ver}")

cuda_ok = cuda_avail
if not cuda_ok:
    print("[FAIL-CLOSED] CELL G BLOCKED. CUDA GPU accelerator unavailable.")
    raise RuntimeError("[FATAL FAIL-CLOSED] CUDA GPU hardware accelerator is unavailable!")

# ------------------------------------------------------------
# PART 2 — FINAL MODEL DIRECTORY & REQUIRED FILES
# ------------------------------------------------------------
model_dir_ok = os.path.exists(DRIVE_MODEL_DIR)
required_files = [
    "model.safetensors",
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json"
]

files_exist = {}
files_non_zero = {}
for rf in required_files:
    rf_path = os.path.join(DRIVE_MODEL_DIR, rf)
    ex = os.path.exists(rf_path)
    files_exist[rf] = ex
    files_non_zero[rf] = ex and (os.path.getsize(rf_path) > 0)

config_ok           = files_non_zero.get("config.json", False)
generation_config_ok= files_non_zero.get("generation_config.json", False)
tokenizer_json_ok   = files_non_zero.get("tokenizer.json", False)
tokenizer_config_ok = files_non_zero.get("tokenizer_config.json", False)

# ------------------------------------------------------------
# PART 3 — SPECIAL TOKENIZER METADATA REPAIR
# ------------------------------------------------------------
sp_tok_path = os.path.join(DRIVE_MODEL_DIR, "special_tokens_map.json")
if not os.path.exists(sp_tok_path) or os.path.getsize(sp_tok_path) == 0:
    print("\n[REPAIR] special_tokens_map.json missing or 0 bytes. Regenerating using tokenizer save_pretrained()...")
    try:
        try:
            repair_tok = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR)
        except Exception:
            repair_tok = AutoTokenizer.from_pretrained("google/flan-t5-small")
        repair_tok.save_pretrained(DRIVE_MODEL_DIR)
        print(f"[REPAIR SUCCESS] special_tokens_map.json regenerated ({os.path.getsize(sp_tok_path):,} bytes)")
    except Exception as e:
        print(f"[REPAIR FAIL] Failed to regenerate special_tokens_map.json: {e}")

special_tokens_map_ok = os.path.exists(sp_tok_path) and (os.path.getsize(sp_tok_path) > 0)
special_tokens_json_ok = False
if special_tokens_map_ok:
    try:
        with open(sp_tok_path, "r", encoding="utf-8") as f:
            json.load(f)
        special_tokens_json_ok = True
    except Exception:
        special_tokens_json_ok = False

# ------------------------------------------------------------
# PART 4 — OUTPUT DIRECTORY
# ------------------------------------------------------------
docs_dir = os.path.join(DRIVE_ROOT, "docs") if "DRIVE_ROOT" in locals() else "/content/drive/MyDrive/AQPG/docs"
os.makedirs(docs_dir, exist_ok=True)
output_dir_ok = os.path.exists(docs_dir) and os.access(docs_dir, os.W_OK)

# ------------------------------------------------------------
# PART 5 — EVALUATION OUTPUT PARENT DIRECTORIES
# ------------------------------------------------------------
drive_base = DRIVE_ROOT if "DRIVE_ROOT" in locals() else "/content/drive/MyDrive/AQPG"
eval_output_paths = [
    os.path.join(drive_base, "phase20_v16_generated_outputs.jsonl"),
    os.path.join(drive_base, "phase20_v16_evaluation_summary.json"),
    os.path.join(drive_base, "phase20_quality_evaluation.json"),
    os.path.join(drive_base, "phase20_failure_analysis.json"),
    os.path.join(drive_base, "phase20_subject_confusion_matrix.json"),
    os.path.join(drive_base, "phase20_control_sensitivity.json"),
    os.path.join(drive_base, "phase20_template_diversity.json"),
    os.path.join(drive_base, "phase20_numerical_evaluation.json"),
    os.path.join(drive_base, "phase20_memorization.json"),
    os.path.join(drive_base, "phase20_checkpoint_verification.json"),
    os.path.join(docs_dir, "phase20_step9_evaluation_report.md")
]

for p_out in eval_output_paths:
    os.makedirs(os.path.dirname(os.path.abspath(p_out)), exist_ok=True)

# ------------------------------------------------------------
# PART 6 — MODEL WEIGHT INTEGRITY
# ------------------------------------------------------------
EXPECTED_MODEL_SIZE = 307867048
EXPECTED_MODEL_SHA  = "E29B95E18DFE5CF6D8DB2D1F8781843C9DD4B8EE2CE81C9ED26E8DF56F443BC4"

model_weights_path = os.path.join(DRIVE_MODEL_DIR, "model.safetensors")
model_weights_ok   = os.path.exists(model_weights_path) and (os.path.getsize(model_weights_path) == EXPECTED_MODEL_SIZE)

def compute_file_sha256(filepath):
    sha = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            sha.update(chunk)
    return sha.hexdigest().upper()

model_sha_val = compute_file_sha256(model_weights_path) if os.path.exists(model_weights_path) else ""
model_sha_ok  = (model_sha_val == EXPECTED_MODEL_SHA)

# ------------------------------------------------------------
# PART 7 — MODEL LOAD TEST
# ------------------------------------------------------------
EXPECTED_PARAM_COUNT = 76961152
model_reload_ok    = False
parameter_count_ok = False
nan_inf_ok         = False
actual_param_count = 0
nan_params_count   = 0
inf_params_count   = 0

try:
    reload_tok   = AutoTokenizer.from_pretrained(DRIVE_MODEL_DIR, local_files_only=True)
    reload_model = AutoModelForSeq2SeqLM.from_pretrained(DRIVE_MODEL_DIR, local_files_only=True)
    model_reload_ok = True
    
    for p in reload_model.parameters():
        actual_param_count += p.numel()
        if torch.isnan(p).any(): nan_params_count += 1
        if torch.isinf(p).any(): inf_params_count += 1
        
    parameter_count_ok = (actual_param_count == EXPECTED_PARAM_COUNT)
    nan_inf_ok         = (nan_params_count == 0 and inf_params_count == 0)
except Exception as e:
    print(f"[MODEL LOAD FAIL] Error: {e}")

# ------------------------------------------------------------
# PART 8 — PROMPT INTEGRITY
# ------------------------------------------------------------
EXPECTED_PROMPT_COUNT = 520
EXPECTED_PROMPT_SHA   = "91335C1EC938454BADFD551975018689A2B87489EA10BECDD05C134A1ED3082E"

prompts_path = PROMPTS_FILE if "PROMPTS_FILE" in locals() and os.path.exists(PROMPTS_FILE) else "/content/drive/MyDrive/AQPG/backend/ml/evaluation/phase20_evaluation_prompts.jsonl"
if not os.path.exists(prompts_path):
    prompts_path = "phase20_evaluation_prompts.jsonl"

prompts_ok = os.path.exists(prompts_path)

actual_prompt_count = 0
if prompts_ok:
    with open(prompts_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                actual_prompt_count += 1

prompt_count_ok = (actual_prompt_count == EXPECTED_PROMPT_COUNT)
actual_prompt_sha = compute_file_sha256(prompts_path) if prompts_ok else ""
prompt_sha_ok   = (actual_prompt_sha == EXPECTED_PROMPT_SHA)

# ------------------------------------------------------------
# PART 9 — EVALUATOR EXISTENCE
# ------------------------------------------------------------
eval_script_path = EVAL_SCRIPT if "EVAL_SCRIPT" in locals() and os.path.exists(EVAL_SCRIPT) else "backend/ml/evaluation/evaluate_flan_t5_v16.py"
evaluator_ok = os.path.exists(eval_script_path) and (os.path.getsize(eval_script_path) > 0)

# ------------------------------------------------------------
# PART 10 — FINAL GATE
# ------------------------------------------------------------
ready_for_cell_g = (
    cuda_ok and
    model_dir_ok and
    model_weights_ok and
    model_sha_ok and
    config_ok and
    generation_config_ok and
    tokenizer_json_ok and
    tokenizer_config_ok and
    special_tokens_map_ok and
    special_tokens_json_ok and
    model_reload_ok and
    parameter_count_ok and
    nan_inf_ok and
    prompts_ok and
    prompt_count_ok and
    prompt_sha_ok and
    evaluator_ok and
    output_dir_ok
)

if not ready_for_cell_g:
    print("\n[FAIL-CLOSED] CELL G BLOCKED.")
    raise RuntimeError("[FATAL FAIL-CLOSED] Step 9 pre-evaluation gate checks failed! Inspect log above.")

print()
print("============================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL F")
print("STEP 9 PRE-EVALUATION GATE")
print("============================================================")
print()
print("CUDA:                         PASS")
print(f"GPU:                          {gpu_name}")
print("FINAL MODEL:                  PASS")
print("MODEL SIZE:                   307,867,048 bytes")
print("MODEL SHA-256:                PASS")
print("PARAMETER COUNT:              76,961,152")
print("NaN PARAMETERS:               0")
print("Inf PARAMETERS:               0")
print("MODEL RELOAD:                 PASS")
print()
print("CONFIG.JSON:                  PASS")
print("GENERATION_CONFIG.JSON:       PASS")
print("TOKENIZER.JSON:               PASS")
print("TOKENIZER_CONFIG.JSON:        PASS")
print("SPECIAL_TOKENS_MAP.JSON:      PASS")
print("SPECIAL_TOKENS_MAP JSON:      PASS")
print()
print("EVALUATION SCRIPT:            PASS")
print("PROMPT FILE:                  PASS")
print("PROMPT COUNT:                 520")
print("PROMPT SHA-256:               PASS")
print()
print("OUTPUT DIRECTORY:             PASS")
print()
print("============================================================")
print("READY FOR CELL G: TRUE")
print("============================================================")
print()
print("CELL F COMPLETE.")
print()
print("CELL G is now authorized to run the 520-prompt evaluation.")
print()
print("============================================================")


## CELL G — STEP 9: 520 CONTROLLED PROMPT POST-TRAINING EVALUATION

**Purpose:** Execute deterministic 520-prompt evaluation script against the verified final V16 model on GPU.

> [!IMPORTANT]
> **Evaluation Boundary:** Uses the unchanged 520-prompt set and verified final V16 model checkpoint on Drive.

In [ ]:
print("====================================================================")
print("AQPG PHASE 20 V16 PIPELINE — CELL G: 520 PROMPT EVALUATION")
print("====================================================================")

print(f"FINAL MODEL SHA-256: {drive_model_sha256}")
print(f"FINAL MODEL SIZE:    {drive_model_size:,} bytes")
print(f"PROMPT SHA-256:      {prompt_sha}")
print(f"PROMPT COUNT:        {prompt_count}")
print(f"GPU NAME:            {gpu_name}")
print()

# Execute evaluate_flan_t5_v16.py
!python backend/ml/evaluation/evaluate_flan_t5_v16.py --checkpoint_dir /content/drive/MyDrive/AQPG/backend/ml/models/checkpoints/flan_t5_v16_small/
